In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score 

# Models
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier





# Imbalance
from imblearn.over_sampling import RandomOverSampler

In [ ]:
models = {
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Extra Trees": ExtraTreesClassifier(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss',tree_method="hist",device="cuda") 
}

In [ ]:
DATA_DIR = r"D:\Gait Dataset\Code\Koya_Gait_Combined_12Feature_Output"
csv_file = os.path.join(DATA_DIR, "Koya_Gait_Combined_12Features.csv")

# Optional safety checks
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(f"Folder not found: {DATA_DIR}")
if not os.path.isfile(csv_file):
    raise FileNotFoundError(f"CSV file not found: {csv_file}")
if not os.access(DATA_DIR, os.R_OK):
    raise PermissionError(f"No read permission for folder: {DATA_DIR}")

df = pd.read_csv(csv_file)
print(df.head())

In [ ]:
df.columns

In [ ]:
X = df.drop("Label", axis=1)
y = df["Label"]

#  FIX LABELS
y = y - 1


In [ ]:
X = X.fillna(X.mean())

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Handle imbalance
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

train_test_results = {}

print("\n===== TRAIN-TEST RESULTS =====")

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    train_test_results[name] = acc

    print(f"{name}: {acc:.4f}")


In [ ]:
# 6. K-FOLD VALIDATION

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

kfold_results = {}

print("\n===== K-FOLD RESULTS =====")

for name, model in models.items():
    accuracies = []

    for train_idx, test_idx in skf.split(X_scaled, y):
        X_train_k, X_test_k = X_scaled[train_idx], X_scaled[test_idx]
        y_train_k, y_test_k = y.iloc[train_idx], y.iloc[test_idx]

        # Handle imbalance in each fold
        X_train_k_res, y_train_k_res = ros.fit_resample(X_train_k, y_train_k)

        model.fit(X_train_k_res, y_train_k_res)
        y_pred_k = model.predict(X_test_k)

        acc_k = accuracy_score(y_test_k, y_pred_k)
        accuracies.append(acc_k)

    avg_acc = np.mean(accuracies)
    kfold_results[name] = avg_acc

    print(f"{name}: {avg_acc:.4f}")


In [ ]:
comparison_df = pd.DataFrame({
    "Model": list(models.keys()),
    "Train-Test Accuracy": [train_test_results[m] for m in models.keys()],
    "K-Fold Accuracy": [kfold_results[m] for m in models.keys()]
})

print("\n===== FINAL COMPARISON =====")
print(comparison_df)

In [ ]:
comparison_df.set_index("Model").plot(kind='bar')
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xticks(rotation=45)
plt.show()

In [ ]:
from sklearn.feature_selection import RFE
extra_tree = ExtraTreesClassifier(n_estimators=100, random_state=42)
feature_names = X.columns
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names)
rfe = RFE(
    estimator=extra_tree,
    n_features_to_select=10,   # 🔥 exactly 10 features
    step=1
)

rfe.fit(X_scaled_df, y)

selected_features = X_scaled_df.columns[rfe.support_]

print("Selected Features (exactly 10):")
print(list(selected_features))

In [ ]:
X_scaled_df

In [ ]:
X_selected = X_scaled_df[selected_features]


In [ ]:
X_selected


In [ ]:

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n===== K-FOLD RESULTS ON COMMON FEATURES =====")

for name, model in models.items():
    accuracies = []

    for train_idx, test_idx in skf.split(X_selected, y):
        X_train = X_selected.iloc[train_idx]
        X_test = X_selected.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        # Handle imbalance inside each fold
        ros = RandomOverSampler(random_state=42)
        X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

        model.fit(X_train_res, y_train_res)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)

    print(f"{name}: {np.mean(accuracies):.4f}")

In [ ]:
X_selected = X_scaled_df[common_features]
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import RandomOverSampler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# Handle imbalance
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

models = {
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(),
    "Decision Tree": DecisionTreeClassifier(),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Extra Trees": ExtraTreesClassifier(),
    "XGBoost": XGBClassifier(eval_metric='mlogloss')
}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc:.4f}")